In [ ]:
%load_ext watermark
%watermark -a "Paul A. Wambo" -v -p Bio,matplotlib,numpy,pandas,pyMIBiG,seaborn,sklearn,transformers,torch

c:\Users\Yvael\Desktop\polo\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Author: Paul A. Wambo

Python implementation: CPython
Python version       : 3.11.4
IPython version      : 9.7.0

Bio         : 1.86
matplotlib  : 3.10.7
numpy       : 2.3.5
pandas      : 2.3.3
pyMIBiG     : not installed
seaborn     : 0.13.2
sklearn     : 1.7.2
transformers: 4.57.1
torch       : 2.9.1+cu126



In [ ]:
import os
import json

import pandas as pd

In [ ]:
basedir = "C:/Users/Yvael/Desktop/polo/data/input/mibig4"
datafiles = [os.path.join(basedir, file) for file in os.listdir(basedir) if file.endswith(".json")]
print(f"Found {len(datafiles)} data files.")

Found 3013 data files.


In [ ]:
df = pd.DataFrame(columns=[
    "accession",
    "compound_name",
    "quality",
    "status",
    "completeness",
    "class",
    "subclass",
    "compound_structure",
    "npatlas",
    "pubchem",
    "chembl",
])

In [ ]:
with open(datafiles[0], "r") as file:
    mibig_data = json.load(file)

In [ ]:
def process_file(datafile):
    with open(datafile, "r") as file:
        mibig_data = json.load(file)

    # Safely get compounds
    compounds = mibig_data.get("compounds", [])
    if not isinstance(compounds, list) or len(compounds) == 0:
        compound0 = {}
    else:
        compound0 = compounds[0]

    # Safely get database IDs
    ids = compound0.get("databaseIds")
    if not ids:
        id_dict = {"npatlas": None, "pubchem": None, "chembl": None}
    else:
        # protect against malformed entries
        try:
            id_dict = {k: v for k, v in (item.split(":") for item in ids)}
        except Exception:
            id_dict = {"npatlas": None, "pubchem": None, "chembl": None}

    return {
        "accession": mibig_data.get("accession"),
        "compound_name": compound0.get("name"),
        "quality": mibig_data.get("quality"),
        "status": mibig_data.get("status"),
        "completeness": mibig_data.get("completeness"),
        "compound_structure": compound0.get("structure"),
        "class": mibig_data['biosynthesis']['classes'][0].get("class"),
        "subclass": mibig_data['biosynthesis']['classes'][0].get("subclass"),
        "npatlas": id_dict.get("npatlas"),
        "pubchem": id_dict.get("pubchem"),
        "chembl": id_dict.get("chembl"),
    }

In [ ]:
from tqdm import tqdm

rows = []
for datafile in tqdm(datafiles, desc="Processing files"):
    row = process_file(datafile)
    rows.append(row)

df = pd.DataFrame(rows)
df.head()

Processing files: 100%|██████████| 3013/3013 [00:16<00:00, 182.04it/s]


,accession,compound_name,quality,status,completeness,compound_structure,class,subclass,npatlas,pubchem,chembl
0,BGC0000001,abyssomicin C,questionable,active,complete,CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\C=C/C(=O)[C@@H]...,PKS,Type I,NPA01961,71455791,CHEMBL1651097
1,BGC0000002,aculeximycin,questionable,active,unknown,CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(C)...,PKS,Unknown,NPA09222,6449919,None
2,BGC0000003,AF-toxin,questionable,active,unknown,CCC(C)C(C(=O)OC(/C=C/C=C/C=C/C(=O)O)C1(CO1)C)O...,PKS,Unknown,None,6442850,None
3,BGC0000004,aflatoxin G1,questionable,active,unknown,COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4[C@H]5C=CO[C@...,PKS,Unknown,NPA014300,14421,None
4,BGC0000005,aflatoxin,questionable,retired,unknown,COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4C5C=COC5OC4=C1,PKS,Unknown,None,14421,None


In [ ]:
outpath = "C:/Users/Yvael/Desktop/polo/data/output"
df.to_parquet(os.path.join(outpath, "mibig4_compound_data.parquet"), engine="pyarrow", index=False)